In [1]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import os, pickle
from collections import Counter

In [2]:
_RESEARCH_PRODUCTS_FILTERED_ = "./output/products/research_products_filtered.json"
_CITATION_MERGED_ = "./output/citations/citations_merged.json"
_OUTPUT_GRAPH_ = "./output/graphs/"
os.makedirs(_OUTPUT_GRAPH_, exist_ok = True)

In [3]:
# Load filtered dataset
with open(_RESEARCH_PRODUCTS_FILTERED_, "r", encoding="utf-8") as f:
    filtered_data = json.load(f)

# Load citation set
with open(_CITATION_MERGED_, "r", encoding="utf-8") as f:
    citation_set = json.load(f)

# Print dataset sizes
print(f"Filtered dataset entries: {len(filtered_data)}")
print(f"Citation set entries:     {len(citation_set)}")

# Build a mapping from id to data for fast lookup
id_to_data = {item["id"]: item for item in filtered_data}

# Create a directed graph
G = nx.DiGraph()

# Add all UNIPI nodes with their attributes
for node_id, data in id_to_data.items():
    G.add_node(node_id, **data, is_unipi=True)

# Add citation edges and any external nodes
for source, target in citation_set:
    if source not in G:
        G.add_node(source, is_unipi=False)
    if target not in G:
        G.add_node(target, is_unipi=False)
    G.add_edge(source, target)

# Print graph statistics
print("\nGraph (nodes, edges) = ({}, {})".format(G.number_of_nodes(), G.number_of_edges()))

Filtered dataset entries: 264930
Citation set entries:     8271912

Graph (nodes, edges) = (4989602, 8271912)


In [ ]:
# Select all UNIPI nodes with valid indicators and citationImpact fields
unipi_nodes = [
    (node, data)
    for node, data in G.nodes(data=True)
    if data.get("is_unipi", False)
    and isinstance(data.get("indicators"), dict)
    and isinstance(data["indicators"].get("citationImpact"), dict)
]

# Extract indegree and citation counts
indegree_list = [G.in_degree(node) for node, _ in unipi_nodes]
citation_count_list = [data["indicators"]["citationImpact"].get("citationCount", 0) for _, data in unipi_nodes]

# Scatter plot
plt.figure(figsize=(4, 4))
plt.scatter(citation_count_list, indegree_list, alpha=0.5)
plt.xlabel("citationCount (dataset)")
plt.ylabel("indegree (graph)")
plt.title("Indegree vs CitationCount (UniPi)")
plt.grid(True)
plt.show()

# Mean absolute error
gaps = np.array([ind - cit for ind, cit in zip(indegree_list, citation_count_list)])
print(f"Mean absolute error between indegree and citationCount: {np.mean(np.abs(gaps)):.2f}")

In [ ]:
# Drop nodes with no links
isolated = list(nx.isolates(G))
G.remove_nodes_from(isolated)
print(f"Number of isolated nodes removed: {len(isolated)}")

# Define node type helper
get_node_type = lambda n: "UNIPI" if G.nodes[n].get("is_unipi", False) else "EXT"

# Count nodes by type
node_counts = Counter(get_node_type(n) for n in G.nodes())
print(f"Nodi UNIPI: {node_counts.get('UNIPI', 0)}")
print(f"Nodi EXT:   {node_counts.get('EXT', 0)}")
print(f"Totale:     {G.number_of_nodes()}")

# Count edges by type combination
edge_counts = Counter((get_node_type(u), get_node_type(v)) for u, v in G.edges())
total_edges = G.number_of_edges() or 1

print("\nEdges grouped by node type per combination:")
for (src, tgt), count in sorted(edge_counts.items()):
    pct = count / total_edges * 100
    print(f"{src:5} -> {tgt:5}: {count:6} ({pct:.2f}%)")
print(f"{'Totale':14}: {G.number_of_edges()}")

In [ ]:
path = os.path.join(_OUTPUT_GRAPH_, "citation_graph_not_cleaned.gpickle")

with open(path, "wb") as f:
    pickle.dump(G, f)